# EA3 - Observabilidad con Grafana (servicios locales vs nube)

## Objetivos
- Ver los servicios del entorno **funcionando en tiempo real** en Grafana.
- Entender la observabilidad como otro servicio que se opera en local, con su
  equivalente **gestionado** en la nube.
- Recorrer el pipeline `Kafka → Spark Streaming → Postgres → Grafana`.

> **Requiere el perfil `completo`:** `docker compose --profile completo up -d`

## El stack de observabilidad

| Pieza local | Rol | GCP | AWS | Azure |
|---|---|---|---|---|
| **Grafana** | Dashboards | Cloud Monitoring / Looker Studio | CloudWatch / QuickSight | Azure Monitor / Power BI |
| **Prometheus** | Métricas (TSDB) | Managed Prometheus | Amazon Managed Prometheus | Azure Monitor |
| **cAdvisor** | CPU/RAM por contenedor | métricas de GKE | ECS/EKS | AKS |
| **kafka-exporter** | Lag/throughput de Kafka | métricas de Pub/Sub | Kinesis / MSK | Event Hubs |
| **Postgres** | Serving layer (datos) | BigQuery | Redshift | Synapse |
| **Spark Streaming** | Procesamiento | Dataflow | Kinesis Analytics | Stream Analytics |

**Idea fuerza:** en local *vos* operás Prometheus, los exporters y Grafana. En la
nube todo eso es gestionado: pagás por uso y el proveedor lo mantiene. Los
conceptos (métricas, lag, dashboards) son idénticos.

## 1. Abrí Grafana

En el navegador: **http://localhost:3000** (o el puerto que pusiste en
`GRAFANA_PORT` dentro de `.env`). Entrás directo (acceso anónimo de lectura).

En la carpeta **Big Data** vas a ver dos dashboards:
- **🩺 Infraestructura** — CPU/RAM de cada contenedor, estado up/down, Kafka.
- **📊 Negocio en vivo** — ventas por región, throughput, monto acumulado.

El de negocio estará vacío hasta que generes datos (pasos 2 y 3).

## 2. Generá transacciones hacia Kafka

La siguiente celda lanza el generador en segundo plano (60s de eventos).

In [ ]:
import subprocess, sys

# Produce transacciones al topic 'transacciones' durante 60s, en segundo plano.
proc = subprocess.Popen([
    sys.executable, "/home/jovyan/scripts/generar_datos_streaming.py",
    "--tipo", "transacciones", "--velocidad", "8",
    "--duracion", "60", "--topic", "transacciones",
])
print("Generador lanzado (PID", proc.pid, ") — produciendo a 'transacciones' por 60s.")

## 3. Arrancá el job de streaming Kafka → Postgres

Este job consume el topic, agrega por región y escribe en Postgres (lo que lee
Grafana). **Queda corriendo**: cuando quieras frenarlo, interrumpí el kernel
(botón ■ / Kernel → Interrupt).

> Mientras corre, mirá el dashboard **📊 Negocio en vivo** en Grafana: se va
> actualizando solo cada pocos segundos.

In [ ]:
# Ejecuta el job de streaming (bloqueante; interrumpí el kernel para frenar).
%run /home/jovyan/scripts/streaming_a_postgres.py

---
## Ejercicio 1 — Panel "Top 5 productos por monto"

En el dashboard de **Negocio**, los datos se agregan por *región*, no por
*producto*. Para tener "top productos":

1. Modificá `scripts/streaming_a_postgres.py` para agrupar también por
   `producto` (o creá una tabla nueva `ventas_producto`).
2. En Grafana → dashboard Negocio → **Add panel** → tipo *Bar chart* → datasource
   **Postgres-Analytics** → query:
   ```sql
   SELECT producto, SUM(monto_total) AS monto
   FROM ventas_producto
   GROUP BY producto ORDER BY monto DESC LIMIT 5;
   ```
3. Guardá el panel y observá cómo se llena al correr el job.

**Símil cloud:** esto mismo lo harías en Looker Studio / QuickSight sobre
BigQuery / Redshift.

## Ejercicio 2 — Panel de memoria de Kafka (infraestructura)

En el dashboard de **Infraestructura**:

1. **Add panel** → tipo *Time series* → datasource **Prometheus**.
2. Query (PromQL):
   ```promql
   container_memory_usage_bytes{name="bigdata-kafka"}
   ```
3. Unidad: *bytes*. Guardá y observá la memoria del contenedor de Kafka.

**Símil cloud:** es la misma métrica que verías de un broker gestionado en
CloudWatch / Cloud Monitoring / Azure Monitor.

---
## Cierre — ¿quién opera qué?

| | Local (este entorno) | Nube |
|---|---|---|
| Instalar/actualizar Prometheus, Grafana | **Vos** | El proveedor |
| Escalar el almacenamiento de métricas | **Vos** | Automático |
| Construir dashboards | Igual (Grafana/Looker/QuickSight) | Igual |
| Conceptos (métricas, lag, paneles) | **Idénticos** | **Idénticos** |

La observabilidad es transversal: una vez que entendés qué medir (CPU, memoria,
lag, throughput) y cómo visualizarlo, la herramienta concreta —Grafana o el
servicio gestionado de la nube— es secundaria.